# Case Study 4: Environmental accounting

This notebook demonstrates a reproducible calculation using **synthetic records and illustrative factors**. It is not a verified emissions inventory or certification assessment. Run from the project root.

## 1. Generate and inspect the sample

The sample contains 2,000 records and injected site, percentage, Scope 1, audit status, and duplicate-ID defects.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'analyze.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from generate_sample import main as generate_sample
from analyze import process
generate_sample()
raw = pd.read_csv(ROOT / 'data/raw/synthetic_facility_logs.csv')
print(raw.shape)
print(raw.head().to_string(index=False))

## 2. Clean and calculate

`process` writes every rejected record and its reason. It uses four separately versioned assumptions in `factors.csv`. Scope 1 is provided as a direct-emission source field; Scope 2 and the limited Scope 3 categories are calculated from activity data. No invalid value is silently capped or made positive.

In [ ]:
control = process(ROOT / 'data/raw/synthetic_facility_logs.csv', ROOT / 'results')
assert control['reconciles']
exceptions = pd.read_csv(ROOT / 'results/exceptions.csv')
print(exceptions['issue'].value_counts().to_string())

## 3. Inspect the equations

All results are metric tons CO2e. Scope 2 location based = gross electricity kWh × electricity factor ÷ 1,000. Limited Scope 3 = water m3 × water factor ÷ 1,000 + recycled waste kg × recycled factor ÷ 1,000 + landfill waste kg × landfill factor ÷ 1,000. The certificate adjusted number is a sensitivity only and cannot be used as a GHG Protocol market based total without quality evidence.

In [ ]:
calc = pd.read_csv(ROOT / 'results/clean_calculations.csv')
f = pd.read_csv(ROOT / 'factors.csv').set_index('factor_id')['kgco2e_per_unit']
r = calc.iloc[0]
assert abs(r.scope2_location_tco2e - r.electricity_kwh * f['ELEC_DEMO'] / 1000) < 1e-9
assert abs(r.scope3_limited_tco2e - (r.water_m3*f['WATER_DEMO'] + r.waste_recycled_kg*f['RECYCLE_DEMO'] + r.waste_landfill_kg*f['LANDFILL_DEMO'])/1000) < 1e-9
print(calc[['record_id','site','scope1_tco2e','scope2_location_tco2e','scope3_limited_tco2e','total_limited_tco2e']].head().to_string(index=False))

## 4. Site tracker and quality gate

The sum of accepted row emissions must match the site summary. Report status remains illustrative until source records, boundaries, site factors, instrument evidence, and Scope 3 completeness are reviewed.

In [ ]:
site = pd.read_csv(ROOT / 'results/site_summary.csv')
assert abs(calc.total_limited_tco2e.sum() - site.total_limited_tco2e.sum()) < 1e-8
print(site[['site','electricity_kwh','water_m3','waste_kg','recycling_rate_pct','scope1_tco2e','scope2_location_tco2e','scope3_limited_tco2e','total_limited_tco2e']].round(2).to_string(index=False))
print('Rejected rows:', len(exceptions))